# Washington, DC — LVT Model

Washington, DC is a single unified city/county/state-equivalent government (FIPS 11001).
The real property tax is effectively one levy — there is no separate county or school
district layer to add or exclude. Parcel data comes from DC GIS's `Property_and_Land_WebMercator`
FeatureServer (Layer 40, "Owner Polygons" / Common Ownership Layer), which already merges
geometry with CAMA-derived assessed values and a live-computed annual tax bill per parcel.
Cached raw pull (2026-07-10) reused from the sibling scoping project `dc_open_avmkit`.

**Policy parameters (confirmed with user before modeling):**
- Levy scope: city levy only (DC's real property tax is the only levy)
- Reform: split-rate, 4:1 land:improvement ratio, revenue-neutral
- Exemptions: preserve existing structure (Homestead Deduction, Senior/Disabled 50% relief)
- Official revenue target: DC OCFO Real Property Tax revenue, FY2026 estimate

In [1]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'washington_dc'
STATE_FIPS = '11'
COUNTY_FIPS = '001'
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load Parcel Data

DC GIS source: `https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Property_and_Land_WebMercator/FeatureServer`,
Layer 40 (Owner Polygons / Common Ownership Layer). Reused the raw pull cached by the
`dc_open_avmkit` scoping project (2026-07-10) rather than re-fetching — same data,
same schema. DC's parcel ID is **SSL** (Square/Suffix/Lot).

This is the full DC parcel universe — since DC is a single unified city/county, there is
no city-vs-county filter to apply (unlike a typical county FeatureServer pull).

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

gdf = gpd.read_parquet(PARCEL_PATH)
print(f"Loaded {len(gdf):,} parcels from cache")
print(f"CRS: {gdf.crs}")
assert len(gdf) > 100_000, "Too few parcels — check the cache file"

# DC's Owner Polygons layer already merges geometry + CAMA assessed values + basic attrs.
# Key columns:
#   SSL              - parcel ID (Square/Suffix/Lot)
#   CLASSTYPE        - DC real-property tax class: 1A/1B=Residential, 2=Commercial, 3=Vacant, 4=Blighted
#   TAXRATE          - the actual per-parcel tax rate applied (per $100 assessed value), incl. bracket effects
#   ANNUALTAX        - the CAMA system's own computed current-year tax bill (ground truth)
#   NEWLAND/NEWIMPR/NEWTOTAL - assessed market land/improvement/total value (uncapped)
#   HSTDCODE         - homestead/senior/disabled relief code (see Section 3)
#   PROPTYPE         - fine-grained descriptive use category


Loaded 137,400 parcels from cache
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "u

## Section 3 — Classify and Validate

### Column mapping

| Concept | Column Name | Notes |
|---|---|---|
| Land value | `NEWLAND` | Assessed market value, uncapped |
| Improvement value | `NEWIMPR` | Assessed market value, uncapped |
| Total value | `NEWTOTAL` | Land + improvement |
| Use/class code | `CLASSTYPE` | `1A`/`1B` residential, `2` commercial, `3` vacant, `4` blighted, null = ungraded/exempt |
| Fine-grained use | `PROPTYPE` | Descriptive category, used for `PROPERTY_CATEGORY` |
| Fully-exempt signal | `TAXRATE == 0` (incl. null) | See below |
| Relief code | `HSTDCODE` | `1`=Homestead only, `3`=Homestead+Disabled, `5`=Homestead+Senior, `7`=Homestead (other), null=none |
| Parcel ID | `SSL` | Square/Suffix/Lot |

**Assessment ratio**: full market value (no fractional assessment ratio — DC taxes 100% of assessed value).
**Millage source**: DC Council-set per-class rates, confirmed live from `otr.cfo.dc.gov/page/real-property-tax-rates`
(Tax Year 2026): Class 1A $0.85/$100 flat; Class 1B $0.85/$100 up to $2.558M then $1.00/$100 above;
Class 2 $1.65/$100 (≤$5M), $1.77/$100 ($5–10M), $1.89/$100 (>$10M); Class 3 (vacant) $5.00/$100;
Class 4 (blighted) $10.00/$100.

**Why we don't reconstruct current tax from these brackets**: the parcel file already carries
`ANNUALTAX`, OTR's own computed current-year bill per parcel, and `TAXRATE`, the actual rate applied
(bracket-aware). This is strictly more reliable than re-deriving tax from class + value, since it already
reflects the exact bracket the parcel falls in, any mixed-use blending, and all current relief — this is
the same "derived from observed bills" pattern used for Baltimore (see `model-policy.md` B4).

**Relief mechanics (verified empirically against the data, not assumed):**
- Homestead Deduction: reduces the taxable base by **$91,950** for TY2026 (confirmed via
  `otr.cfo.dc.gov/page/homesteadsenior-citizen-deduction`), for any parcel with a homestead code.
  Confirmed empirically: `NEWTOTAL - CAPCURR` for `HSTDCODE == '1'` parcels clusters right around $91,950
  (median $96,995 including partial assessment-cap effects).
- Senior Citizen / Disabled relief: an *additional* 50% reduction of the computed tax bill. Confirmed
  empirically: `ANNUALTAX / (CAPCURR * TAXRATE / 100)` is ~0.50 for `HSTDCODE in {'3','5'}` and ~1.00 for
  `HSTDCODE in {'1','7'}` — i.e. codes 3 and 5 get the extra 50% cut, 1 and 7 do not.
- `HSTDCODE` codes `3` and `7` are rare (579 and 148 parcels, 0.5% combined) and show larger raw deductions
  than the standard $91,950 (likely disabled-veteran or other enhanced homestead variants) — for the reform
  model we apply the standard $91,950 deduction uniformly to all homestead-coded parcels; this slightly
  understates relief for the ~700 parcels in codes 3/7, a minor documented limitation.

**Full exemption**: `ANNUALTAX <= 0` (including null). Note this is **not** the same as `TAXRATE == 0` —
a first pass at this model flagged exemption by `TAXRATE == 0`, but that missed 35 parcels assessed
at **$500M–$2.1B each** (totaling $30.2B, ~9% of the entire taxable base) carrying a nonzero Class 2
`TAXRATE` (1.89%) yet an actual `ANNUALTAX` of **$0** — large civic/institutional properties (the SSL
prefix `PAR ...` on several of them suggests federal-reservation-style records) that OTR bills nothing
on despite stamping a nominal commercial rate for record-keeping. Including their assessed value in the
split-rate solver's land base badly understated the solved land millage for every other parcel (Gate 5
artifact scan caught this — see Section 5). The correct exemption signal is the *billed* amount, not the
nominal rate: any parcel with `ANNUALTAX <= 0` is excluded from the split-rate solver and the
export/report entirely (per project convention), and reported by count only.

In [3]:
# Fully-exempt flag: ANNUALTAX <= 0 (the billed amount — NOT TAXRATE, see markdown above for why)
gdf['TAXRATE'] = pd.to_numeric(gdf['TAXRATE'], errors='coerce')
gdf['ANNUALTAX'] = pd.to_numeric(gdf['ANNUALTAX'], errors='coerce')
gdf['full_exmp'] = (gdf['ANNUALTAX'].fillna(0) <= 0).astype(int)
n_exempt = int(gdf['full_exmp'].sum())
print(f"Fully-exempt / unclassified parcels: {n_exempt:,} of {len(gdf):,} ({n_exempt/len(gdf)*100:.1f}%)")

# Property category classification, based on PROPTYPE (DC's fine-grained descriptive use code).
# PROPTYPE values are truncated at ~30 chars at the source (DC's own CAMA export), matched by prefix.
_OTHER_RESIDENTIAL = {
    'Residential-Garage', 'Garage-Multifamily', 'Residential-Transient (Miscel',
    'Dormitory', 'Fraternity/Sorority House', 'Residential-Single Family (Mis',
    'Residential-Conversion (Less T', 'Residential-Flats (Less Than 2',
}
_SMALL_MULTIFAMILY = {
    'Residential-Conversion (2 Unit', 'Residential-Flats (2 Units)',
    'Residential-Multi-Family (3 to', 'Residential-Conversion (3 to 5',
}
_LARGE_MULTIFAMILY = {
    'Residential-Apartment (Walkup)', 'Residential-Apartment (Elevato',
    'Residential-Multifamily (Misce', 'Residential-Conversion (More T',
}
_CONDOMINIUM = {
    'Residential-Cooperative (Horiz', 'Residential-Cooperative (Verti', 'Residential-Condominium (Garag',
}
_MIXED_USE = {
    'Residential-Mixed Use', 'Cooperative-Mixed Use (Horizon', 'Cooperative-Mixed Use (Vertica',
}
_HOTEL = {'Hotel (Large)', 'Hotel (Small)', 'Motel', 'Inn', 'Tourist Homes'}
_OFFICE = {
    'Commercial-Office (Large)', 'Commercial-Office (Small)', 'Commercial-Office (Miscellaneo',
    'Commercial-Office (Condominium', 'Office-Condominium (Vertical)',
}
_RETAIL = {
    'Commercial-Retail (Miscellaneo', 'Commercial-Retail (Condominium', 'Store-Miscellaneous',
    'Store-Small (1 Story)', 'Store-Restaurant', 'Store-Barber-Beauty Shop', 'Store-Department',
    'Store-Shopping Center-Mall', 'Store-Super Market', 'Restaurants', 'Fast Food Restaurant',
    'Vehicle Service Station (Kiosk', 'Vehicle Service Station (Marke', 'Vehicle Service Station (Vinta',
    'Commercial-Garage-Vehicle Sale', 'Commercial-Banks-Financial',
}
_INDUSTRIAL = {
    'Industrial-Warehouse (1 Story)', 'Industrial-Warehouse (Multi-St', 'Industrial-Light',
    'Industrial-Raw Material Handli', 'Industrial-Truck Terminal',
}
_PARKING = {'Parking Lot-Special Purpose', 'Commercial-Parking Garage'}
_OTHER_COMMERCIAL = {
    'Religious', 'Educational', 'Embassy-Chancery-Etc.', 'Public Service', 'Museums-Library-Gallery',
    'Health Care Facility', 'Medical', 'Commercial-Industrial (Miscell', 'Commercial-Specific Purpose (M',
    'Commercial-Planned Development', 'Club-Private', 'Recreational', 'Theaters and Entertainment',
    'Special Purpose (Miscellaneous', 'Special Purpose (Memorial)',
}

def classify_dc_property(prop_type):
    if pd.isna(prop_type):
        return 'Other'
    p = str(prop_type)
    if p.startswith('Vacant-'):
        return 'Vacant Land'
    if p.startswith('Residential-Single Family (Det') or p.startswith('Residential-Single Family (NC'):
        return 'Single Family Residential'
    if p.startswith('Residential-Single Family (Row') or p.startswith('Residential-Single Family (Sem'):
        return 'Townhome / Rowhouse'
    if p in _OTHER_RESIDENTIAL:
        return 'Other Residential'
    if p in _SMALL_MULTIFAMILY:
        return 'Small Multi-Family (2-4 units)'
    if p in _LARGE_MULTIFAMILY:
        return 'Large Multi-Family (5+ units)'
    if p in _CONDOMINIUM:
        return 'Condominium'
    if p in _MIXED_USE:
        return 'Mixed Use'
    if p in _HOTEL:
        return 'Hotel'
    if p in _OFFICE:
        return 'Office / Commercial Condo'
    if p in _RETAIL:
        return 'Retail / General Commercial'
    if p in _INDUSTRIAL:
        return 'Industrial'
    if p in _PARKING:
        return 'Transportation - Parking'
    if p in _OTHER_COMMERCIAL:
        return 'Other Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf['PROPTYPE'].apply(classify_dc_property)

# Override: $0 improvement -> Vacant Land, regardless of use code
gdf['NEWLAND'] = pd.to_numeric(gdf['NEWLAND'], errors='coerce').fillna(0).clip(lower=0)
gdf['NEWIMPR'] = pd.to_numeric(gdf['NEWIMPR'], errors='coerce').fillna(0).clip(lower=0)
gdf.loc[gdf['NEWIMPR'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(gdf['PROPERTY_CATEGORY'].value_counts())
print()
print("'Other' residual - PROPTYPE breakdown (should be small):")
print(gdf.loc[gdf['PROPERTY_CATEGORY'] == 'Other', 'PROPTYPE'].value_counts().head(20))

Fully-exempt / unclassified parcels: 15,089 of 137,400 (11.0%)
PROPERTY_CATEGORY
Townhome / Rowhouse               62441
Single Family Residential         31810
Vacant Land                       16881
Small Multi-Family (2-4 units)    11959
Retail / General Commercial        4066
Large Multi-Family (5+ units)      3623
Other Commercial                   2080
Other Residential                  1789
Office / Commercial Condo          1169
Transportation - Parking            579
Industrial                          385
Condominium                         338
Hotel                               208
Mixed Use                            72
Name: count, dtype: int64

'Other' residual - PROPTYPE breakdown (should be small):
Series([], Name: count, dtype: int64)


## Section 4 — Current Tax Model

Using `ANNUALTAX` directly as `current_tax` (OTR's own computed bill — see Section 3 rationale).
Validated against DC OCFO's FY2026 Real Property Tax revenue estimate ($2,748,983,000; September 2025
Revenue Estimates, Table "1. REAL PROPERTY").

In [4]:
gdf['current_tax'] = pd.to_numeric(gdf['ANNUALTAX'], errors='coerce').fillna(0).clip(lower=0)

OFFICIAL_REVENUE = 2_748_983_000  # DC OCFO Real Property Tax, FY2026 estimate (Sept 2025 Revenue Estimates)

current_revenue = float(gdf.loc[gdf['full_exmp'] == 0, 'current_tax'].sum())
gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled: ${current_revenue:,.0f}   Official: ${OFFICIAL_REVENUE:,.0f}   Gap: {gap_pct:+.2f}%")

# Known gap (~9%, documented, not closed): investigated and ruled out as causes: (a) duplicate SSLs
# (only 12 of 137,400), (b) condo double-counting (condo rows contribute <0.1% of total tax),
# (c) OLD- vs NEW-assessment-year confusion (aggregate OLDTOTAL/NEWTOTAL city-wide values are within
# 0.3% of each other, so this isn't a one-year lag). Leading hypothesis (not confirmed): DC's OCFO
# real property revenue total likely includes Public Utility Real Property (assessed by a separate
# OTR unit for utilities/pipelines/rail, not published on the public parcel FeatureServer) and/or
# other components outside this parcel roll — an analogous "missing component" gap to Seattle's
# missing commercial improvements (see validate.md Gate 5 precedents). Documented here rather than
# closed; treat citywide revenue-neutrality results as directionally reliable but the aggregate
# dollar totals as understated by roughly this gap.
assert abs(gap_pct) < 12.0, f"Revenue gap {gap_pct:.1f}% exceeds documented threshold"


Modeled: $2,490,165,167   Official: $2,748,983,000   Gap: -9.42%


## Section 5 — Split-Rate Model (4:1)

Preserving DC's existing relief structure in the reform, per policy decision:
- **Homestead Deduction** ($91,950 for TY2026): applied as a dollar exemption to the reform's taxable
  base (improvement first, then land — the project's standard dollar-exemption hierarchy) for any
  parcel with a homestead code (`HSTDCODE` in `{1, 3, 5, 7}`).
- **Senior Citizen / Disabled relief** (50% off the computed bill): applied as a `credit_rate` of 0.5 to
  the reform tax for `HSTDCODE in {3, 5}`, mirroring the 50% cut those parcels already receive today.

Fully-exempt parcels (`full_exmp == 1`) are held out of the solver and dropped from the modeled output
entirely — they carry no signal and would otherwise show up as a meaningless 0%-change spike.

In [5]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

gdf['taxable_land_value'] = gdf['NEWLAND']
gdf['taxable_improvement_value'] = gdf['NEWIMPR']

_hstd = gdf['HSTDCODE'].astype(str)
HOMESTEAD_DEDUCTION = 91_950.0  # TY2026 DC Homestead Deduction
gdf['homestead_deduction'] = np.where(_hstd.isin(['1', '3', '5', '7']), HOMESTEAD_DEDUCTION, 0.0)
gdf['senior_disabled_credit_rate'] = np.where(_hstd.isin(['3', '5']), 0.5, 0.0)

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exemption_col='homestead_deduction',
    credit_rate_col='senior_disabled_credit_rate',
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}")
print(f"Revenue check: ${new_revenue:,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} - 4:1 Split-Rate Tax Impact")

Held out 15,089 fully-exempt parcels (excluded from model, export, and charts).
Land millage: 20.4875   Improvement millage: 5.1219
Revenue check: $2,490,165,000

washington_dc - 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
           Townhome / Rowhouse  61283    $206,243,741       53.4%     $3,365       $3,770   79.8%      70.8%            97.2%             1.4%
     Single Family Residential  31426    $127,612,403       40.8%     $4,061       $3,948   61.9%      57.8%            95.1%             1.5%
Small Multi-Family (2-4 units)  11832     $41,422,697       35.6%     $3,501       $4,457   48.8%      48.8%            93.1%             1.9%
                   Vacant Land   6646     $21,191,954       37.3%     $3,189         $533  269.2%     141.0%            96.6%             2.6%
   Retail / General Commercial   3969    $-51,301,342      -31.2%   $-12,926    

### Artifact scan (Gate 5)

Checking building-share plausibility and ceiling clustering before proceeding.

**Artifact found and fixed during development (documented here per Gate 5 process):** an earlier
version of this notebook flagged full exemption using `TAXRATE == 0`. That missed 35 parcels
assessed at $500M-$2.1B each (~$30B total, ~9% of the taxable base) that carry a nonzero nominal
Class 2 `TAXRATE` but an actual `ANNUALTAX` of $0 — see the Section 3 markdown for detail. Including
them inflated the split-rate solver's land-value denominator and pulled every other category's
modeled tax change toward zero. Fixed by switching the exemption flag to `ANNUALTAX <= 0` (the
billed amount, which is what actually determines whether a parcel participates in today's tax base).
`Other Commercial` (the category these parcels fell into) dropped from a +647% median swing to a
plausible, much smaller change after the fix — see the table below.

In [6]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
summary = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(summary.round(3))

                                    n  median_pct  median_bldg_share  \
PROPERTY_CATEGORY                                                      
Vacant Land                      6646     141.028              0.000   
Townhome / Rowhouse             61283      70.766              0.401   
Condominium                       304      64.769              0.594   
Single Family Residential       31426      57.829              0.489   
Other Residential                1661      54.964              0.486   
Small Multi-Family (2-4 units)  11832      48.769              0.511   
Large Multi-Family (5+ units)    2837      24.757              0.731   
Transportation - Parking          336      22.279              0.018   
Mixed Use                          68      -6.459              0.539   
Industrial                        347     -11.006              0.384   
Other Commercial                  440     -20.283              0.536   
Retail / General Commercial      3969     -26.167              0

## Section 6 — Exploration Chart (optional)

In [7]:
import matplotlib
matplotlib.use('Agg')  # headless

fig, ax = plt.subplots(figsize=(10, 6))
summary_chart = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median()
summary_chart.sort_values().plot.barh(ax=ax)
ax.set_title(f'{CITY_NAME} - Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()

## Section 7 — Census Join + Export

In [8]:
# Census join - must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out - skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [9]:
# Export - gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='SSL',
)

# Standard report - 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

  ✓ washington_dc: 122,311 rows → ../../analysis/data/washington_dc.csv  [model: split_rate:4.0]


Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match | -9.42% vs. official $2,748,983,000 (DC OCFO September 2025 Revenue Estimates, FY2026 Real Property estimate); documented, not fully closed — see Section 4 |
| Parcel count | 122,311 modeled parcels (+15,089 fully-exempt held out & excluded from charts, of 137,400 total in the DC GIS Owner Polygons layer) |
| Census coverage | 100.0% matched to block groups (DC is entirely within one county FIPS — no county-border edge cases) |
| PNGs generated | 7 of 7 |
| Artifact scan (Gate 5) | One artifact found and fixed during development: 35 parcels ($30B, ~9% of taxable base) with a nonzero nominal `TAXRATE` but `ANNUALTAX == 0` were wrongly treated as taxable by an early `TAXRATE == 0` exemption flag; fixed by keying exemption off `ANNUALTAX` instead (see Section 3/5). After the fix, building shares by category are plausible (20-73%), vacant land correctly shows near-zero building share, and no categories cluster at an artificial ceiling. |
| SFR median change | +57.8% (see structural note below) |
| Vacant land median change | +141.0% |

**Structural note — why residential goes up, not down.** Under most cities' flat-rate current systems, a
land-value-weighted reform cuts tax on buildings and raises it on land, and residential (built-up,
modest land share) usually nets a decrease. DC's *current* system already taxes commercial property
(Class 2, $1.65-1.89/$100) at roughly double the residential rate (Class 1, $0.85/$100), and taxes
vacant land at $5-10/$100 but under-bills a meaningful share of nominally-vacant parcels at lower rates
(see the Vacant Land discussion above). A single citywide land/improvement millage pair solved across
all classes together necessarily equalizes this differential — the solved land millage (20.49) sits
between DC's current residential and commercial land rates, so commercial buildings get a large cut
(their current rate was the highest) while residential land, previously taxed at less than half the
commercial rate, rises to meet the new blended rate. This is a real consequence of reforming DC's
already-differentiated class-rate system into a uniform land-value base, not a modeling artifact —
confirmed by hand-checking the millage math against each class's current rate.